# EcoShield AI - Exploratory Data Analysis

Bu notebook, EcoShield AI projesinde kullanılacak **Credit Card Fraud Detection** veri setinin ilk analizini içerir.

Amaç:
- Veri setinin genel yapısını incelemek
- Fraud / normal işlem dağılımını görmek
- Eksik değer ve tekrar eden satırları kontrol etmek
- `Amount` ve `Time` değişkenlerini analiz etmek
- Sprint 1 için görsel ve metinsel çıktı üretmek

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Notebook'un nereden çalıştırıldığına göre veri yolunu otomatik bulalım.
possible_data_paths = [
    Path("data/raw/creditcard.csv"),       # çalışma dizini repo kökü ise
    Path("../data/raw/creditcard.csv"),    # çalışma dizini notebooks klasörü ise
]

RAW_DATA_PATH = None
for path in possible_data_paths:
    if path.exists():
        RAW_DATA_PATH = path
        break

if RAW_DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv bulunamadı. Dosya şu konumda olmalı: data/raw/creditcard.csv"
    )

# Görsel çıktıları kaydedeceğimiz klasörü de çalışma dizinine göre ayarlayalım.
if Path("assets/sprint-1").exists() or Path("assets").exists():
    ASSETS_PATH = Path("assets/sprint-1")
else:
    ASSETS_PATH = Path("../assets/sprint-1")

ASSETS_PATH.mkdir(parents=True, exist_ok=True)

print("EDA başlatıldı.")
print(f"Veri yolu: {RAW_DATA_PATH.resolve()}")
print(f"Çıktı klasörü: {ASSETS_PATH.resolve()}")

## 1. Veriyi Okuma

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

print("Veri başarıyla okundu.")
print(f"Satır sayısı: {df.shape[0]:,}")
print(f"Sütun sayısı: {df.shape[1]:,}")

df.head()

## 2. Genel Veri Bilgisi

In [ ]:
print("Veri seti genel bilgisi:")
df.info()

In [ ]:
print("Sütunlar:")
print(df.columns.tolist())

df.describe().T

## 3. Eksik Değer ve Duplicate Kontrolü

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

if len(missing_values) == 0:
    print("Eksik değer bulunmuyor.")
else:
    print("Eksik değer bulunan sütunlar:")
    display(missing_values)

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Tekrarlı satır sayısı: {duplicate_count:,}")
print(f"Tekrarlı satır oranı: %{duplicate_count / len(df) * 100:.4f}")

## 4. Target Değişkeni ve Class Imbalance Analizi

In [ ]:
target_col = "Class"

class_counts = df[target_col].value_counts().sort_index()
class_ratios = df[target_col].value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "count": class_counts,
    "ratio_percent": class_ratios
})

class_summary.index = ["Normal Transaction", "Fraud Transaction"]

display(class_summary)

In [ ]:
normal_count = int(class_counts.loc[0])
fraud_count = int(class_counts.loc[1])
imbalance_ratio = normal_count / fraud_count

print(f"Normal işlem sayısı: {normal_count:,}")
print(f"Fraud işlem sayısı: {fraud_count:,}")
print(f"Fraud oranı: %{fraud_count / len(df) * 100:.4f}")
print(f"Normal/Fraud oranı: {imbalance_ratio:.2f}")

print("\nYorum:")
print("Veri setinde ciddi bir class imbalance problemi vardır.")
print("Bu nedenle accuracy metriği tek başına yeterli değildir.")
print("Precision, recall, F1-score, ROC-AUC ve PR-AUC metrikleri birlikte değerlendirilmelidir.")

In [ ]:
plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Transaction Count")
plt.xticks(ticks=[0, 1], labels=["Normal", "Fraud"], rotation=0)
plt.tight_layout()

output_path = ASSETS_PATH / "class_distribution.png"
plt.savefig(output_path, dpi=150)
plt.show()

print(f"Grafik kaydedildi: {output_path}")

## 5. Amount Analizi

In [ ]:
amount_summary = df.groupby(target_col)["Amount"].describe()
amount_summary.index = ["Normal Transaction", "Fraud Transaction"]

display(amount_summary)

In [ ]:
plt.figure(figsize=(8, 4))

normal_amount_sample = df[df[target_col] == 0]["Amount"].sample(
    min(5000, (df[target_col] == 0).sum()),
    random_state=42
)

fraud_amount = df[df[target_col] == 1]["Amount"]

normal_amount_sample.hist(bins=50, alpha=0.7, label="Normal Sample")
fraud_amount.hist(bins=50, alpha=0.7, label="Fraud")

plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()

output_path = ASSETS_PATH / "amount_distribution.png"
plt.savefig(output_path, dpi=150)
plt.show()

print(f"Grafik kaydedildi: {output_path}")

## 6. Time Analizi

In [ ]:
time_summary = df.groupby(target_col)["Time"].describe()
time_summary.index = ["Normal Transaction", "Fraud Transaction"]

display(time_summary)

In [ ]:
plt.figure(figsize=(8, 4))

normal_time_sample = df[df[target_col] == 0]["Time"].sample(
    min(5000, (df[target_col] == 0).sum()),
    random_state=42
)

fraud_time = df[df[target_col] == 1]["Time"]

normal_time_sample.hist(bins=50, alpha=0.7, label="Normal Sample")
fraud_time.hist(bins=50, alpha=0.7, label="Fraud")

plt.title("Transaction Time Distribution")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()

output_path = ASSETS_PATH / "time_distribution.png"
plt.savefig(output_path, dpi=150)
plt.show()

print(f"Grafik kaydedildi: {output_path}")

## 7. Feature / Target Ayrımı

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

print("\nTarget dağılımı:")
print(y.value_counts())

## 8. Sprint 1 EDA Özeti

In [ ]:
eda_summary = f"""
EcoShield AI - Sprint 1 EDA Summary

Dataset shape: {df.shape}
Normal transactions: {normal_count}
Fraud transactions: {fraud_count}
Fraud ratio: {fraud_count / len(df) * 100:.4f}%
Missing value columns: {len(missing_values)}
Duplicate rows: {duplicate_count}
Imbalance ratio: {imbalance_ratio:.2f}

Initial observations:
- Dataset is highly imbalanced.
- Accuracy alone is not sufficient for model evaluation.
- Recall, precision, F1-score, ROC-AUC and PR-AUC should be used.
- Class imbalance techniques such as class_weight, SMOTE and threshold tuning should be tested in later sprints.
"""

print(eda_summary)

summary_path = ASSETS_PATH / "eda_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(eda_summary)

print(f"EDA özeti kaydedildi: {summary_path}")

## 9. Oluşturulan Sprint 1 Çıktıları

Bu notebook çalıştırıldığında aşağıdaki dosyalar oluşmalıdır:

```text
assets/sprint-1/class_distribution.png
assets/sprint-1/amount_distribution.png
assets/sprint-1/time_distribution.png
assets/sprint-1/eda_summary.txt
```

In [ ]:
print("Oluşturulan Sprint 1 çıktı dosyaları:")

for file in sorted(ASSETS_PATH.glob("*")):
    print(file)